In [53]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import pytz

In [54]:
apps_df = pd.read_csv(r"C:\Users\ashis\OneDrive\Desktop\GooglePlayProject\Play Store Data.csv")

In [55]:
apps_df = apps_df[apps_df['Installs'] != 'Free']
apps_df['Installs'] = apps_df['Installs'].str.replace('+', '')
apps_df['Installs'] = apps_df['Installs'].str.replace(',', '')
apps_df['Installs'] = pd.to_numeric(
    apps_df['Installs'],
    errors='coerce'
)
apps_df = apps_df.dropna(subset=['Installs'])
apps_df['Installs'] = apps_df['Installs'].astype(int)

In [56]:
apps_df = apps_df[apps_df['Size'].str.contains('M', na=False)]
apps_df['Size'] = apps_df['Size'].str.replace('M', '')
apps_df['Size'] = pd.to_numeric(
    apps_df['Size'],
 errors='coerce')
apps_df = apps_df.dropna(subset=['Size'])

In [57]:
apps_df['Price'] = apps_df['Price'].astype(str)
apps_df['Price'] = apps_df['Price'].str.replace('$', '')
apps_df['Price'] = pd.to_numeric( apps_df['Price'],errors='coerce')
apps_df = apps_df.dropna(subset=['Price'])

In [58]:
apps_df['Revenue'] = apps_df['Installs'] * apps_df['Price']

In [59]:
apps_df['Android Ver'] = apps_df['Android Ver'].astype(str)
apps_df['Android Ver'] = apps_df['Android Ver'].str.extract(r'(\d+\.\d+)')
apps_df['Android Ver'] = pd.to_numeric(
apps_df['Android Ver'],errors='coerce')
apps_df = apps_df.dropna(subset=['Android Ver'])

In [60]:
apps_df['Revenue'] = apps_df['Installs'] * apps_df['Price']

In [61]:
apps_df = apps_df[
    (apps_df['Installs'] > 10000) &
    (apps_df['Revenue'] >= 0) &
    (apps_df['Android Ver'] > 4.0) &
    (apps_df['Size'] > 15) &
    (apps_df['Content Rating'] == 'Everyone') &
    (apps_df['App'].str.len() <= 30)
]

In [62]:
top_categories = apps_df.groupby('Category')['Installs'].sum()
top_categories = top_categories.sort_values(ascending=False).head(3)
top_categories = top_categories.index
top_categories

Index(['GAME', 'FAMILY', 'TOOLS'], dtype='str', name='Category')

In [63]:
apps_df = apps_df[apps_df['Category'].isin(top_categories)]

In [64]:
final_data = apps_df.groupby('Type').agg({
    'Installs': 'mean',
    'Revenue': 'mean'
}).reset_index()
final_data

,Type,Installs,Revenue
0,Free,3.514917e+07,0.000000
1,Paid,2.750000e+05,668083.333333


In [65]:
india = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(india)
current_hour = current_time.hour

In [68]:
if 13 <= current_hour < 14:
    fig3 = make_subplots(
        specs=[[{"secondary_y": True}]]
    )
    fig3.add_trace(
        go.Bar(
            x=final_data['Type'],
            y=final_data['Installs'],
            name='Average Installs'
        ),
        secondary_y=False
    )
    fig3.add_trace(
        go.Scatter(
            x=final_data['Type'],
            y=final_data['Revenue'],
            name='Average Revenue',
            mode='lines+markers'
        ),
        secondary_y=True
    )
    fig3.update_layout(
    title='Installs vs Revenue for Free and Paid Apps',
    xaxis_title='App Type',
    template='plotly_dark',
    width=900,
    height=600
    )
    fig3.update_yaxes(
        title_text='Average Installs',
        secondary_y=False
    )
    fig3.update_yaxes(
        title_text='Average Revenue',
        secondary_y=True
    )
    fig3.show()
else:
    print("Graph available only between 1 PM IST and 2 PM IST")

Graph available only between 1 PM IST and 2 PM IST
